# YOLOWeeds na prática: detecção de plantas daninhas no algodão

Notebook de demonstração que acompanha a apresentação **YOLOWeeds: Visão Computacional Aplicada ao Agronegócio**, baseada em Dang, Chen, Lu e Li (2023), *Computers and Electronics in Agriculture* 205, 107655.

A ideia é reproduzir, em escala reduzida, cada etapa mostrada nos slides:

| Seção | O que faz | Slide |
|---|---|---|
| 1 | Prepara o ambiente | |
| 2 | Baixa uma amostra do CottonWeedDet12 | 5 |
| 3 | Explora as classes e as caixas | 5 e 10 |
| 4 | Mostra o que é data augmentation | 11 |
| 5 | Divide os dados e converte para o formato YOLO | 6 |
| 6 | Treina modelos YOLOv5 de tamanhos diferentes | 6 e 8 |
| 7 | Compara acurácia e velocidade | 8 e 9 |
| 8 | Mede o acerto por classe | 10 |
| 9 | Compara treino com e sem augmentation | 11 |
| 10 | Repete com sementes diferentes (Monte-Carlo) | 6 |
| 11 | Visualiza as predições | 8 |

**Antes de rodar:** em *Ambiente de execução > Alterar o tipo de ambiente de execução*, escolha **GPU (T4)**.

**Importante:** aqui usamos uma amostra pequena e poucas épocas, para caber numa aula. Os números vão ficar abaixo dos do artigo, que usou as 5.648 imagens, 300 épocas e 5 repetições. O objetivo é mostrar o processo e as tendências, não reproduzir os valores exatos.

## 1. Preparar o ambiente

In [ ]:
!pip install -q fiftyone ultralytics

In [ ]:
import os, random, glob, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import torch

print("GPU disponível:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

### Configurações da demonstração

Ajuste aqui o tamanho do experimento. Os valores padrão rodam em cerca de 30 a 40 minutos numa GPU T4 do Colab.

In [ ]:
N_AMOSTRAS = 800        # imagens baixadas do dataset (o completo tem 5.648, cerca de 29 GB)
EPOCAS = 30             # o artigo usou 300
IMGSZ = 640             # tamanho de entrada da rede
BATCH = 16              # mesmo batch do artigo
SEMENTE = 42

MODELOS = ["yolov5nu.pt", "yolov5su.pt"]   # nano e small; acrescente "yolov5mu.pt" para ver um modelo maior

RODAR_AUGMENTATION = True    # seção 9: treina um modelo extra sem augmentation
RODAR_MONTE_CARLO = False    # seção 10: repete o treino com 3 sementes (mais demorado)

CLASSES = ["Waterhemp", "MorningGlory", "Purslane", "SpottedSpurge", "Carpetweed", "Ragweed",
           "Eclipta", "PricklySida", "PalmerAmaranth", "Sicklepod", "Goosegrass", "CutleafGroundcherry"]

APLICAR_EXIF = False   # veja a nota da seção 3 sobre a orientação das fotos

BASE = "/content/yoloweeds"
os.makedirs(BASE, exist_ok=True)
random.seed(SEMENTE); np.random.seed(SEMENTE)

## 2. Baixar o CottonWeedDet12 (slide 5)

O dataset original está no Zenodo (registro 7535814). Aqui usamos a cópia publicada no Hugging Face pela Voxel51, que já traz as caixas organizadas e permite baixar só uma parte.

Licença: CC BY-NC 4.0 (uso não comercial, citando a fonte).

In [ ]:
import fiftyone as fo
import fiftyone.utils.random as four
from fiftyone import ViewField as F
from fiftyone.utils.huggingface import load_from_hub

dataset = load_from_hub("Voxel51/CottonWeedDet12", max_samples=N_AMOSTRAS, name="cottonweeddet12_demo", overwrite=True)
print(dataset)

## 3. Explorar os dados (slides 5 e 10)

Primeiro, os três números do slide 5 na base completa, e a mesma contagem na amostra que baixamos.

In [ ]:
# Contagem de caixas por classe na base completa (cartão do dataset no Hugging Face)
completo = {"Waterhemp": 1959, "MorningGlory": 1341, "Purslane": 992, "Carpetweed": 962, "SpottedSpurge": 952,
            "Ragweed": 896, "Eclipta": 865, "PricklySida": 484, "PalmerAmaranth": 348, "Sicklepod": 241,
            "Goosegrass": 214, "CutleafGroundcherry": 116}

amostra = dataset.count_values("ground_truth.detections.label")
tabela = pd.DataFrame({"caixas_base_completa": pd.Series(completo),
                       "caixas_na_amostra": pd.Series(amostra)}).fillna(0).astype(int)
tabela = tabela.sort_values("caixas_base_completa", ascending=False)

print("Base completa: 5.648 imagens, 12 classes,", sum(completo.values()), "caixas")
print("Amostra:", len(dataset), "imagens,", (tabela.caixas_na_amostra > 0).sum(), "classes,", tabela.caixas_na_amostra.sum(), "caixas")
tabela

In [ ]:
cores = ["#2F5D3A" if c not in ("Carpetweed", "Goosegrass") else "#B5532A" for c in tabela.index]
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(tabela.index, tabela.caixas_base_completa, color=cores)
ax.set_ylabel("Caixas anotadas")
ax.set_title("Caixas por classe no CottonWeedDet12 (base completa)")
plt.xticks(rotation=40, ha="right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

O desequilíbrio é grande: Waterhemp tem quase 17 vezes mais caixas que Cutleaf Groundcherry. Em laranja estão Carpetweed e Goosegrass, os dois pontos cegos do slide 10. Vamos voltar a eles na seção 8.

Agora, um exemplo de cada classe com as caixas desenhadas, como na grade do slide 5.

In [ ]:
def abrir(caminho):
    img = Image.open(caminho)
    return ImageOps.exif_transpose(img) if APLICAR_EXIF else img

def desenhar(ax, amostra, titulo=None, cor="red"):
    img = abrir(amostra.filepath)
    img.thumbnail((900, 900))
    w, h = img.size
    ax.imshow(img); ax.axis("off")
    for d in amostra.ground_truth.detections:
        x, y, bw, bh = d.bounding_box  # coordenadas relativas (0 a 1)
        ax.add_patch(patches.Rectangle((x * w, y * h), bw * w, bh * h, fill=False, edgecolor=cor, linewidth=2))
    if titulo: ax.set_title(titulo, fontsize=11)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, classe in zip(axes.flat, CLASSES):
    vista = dataset.filter_labels("ground_truth", F("label") == classe)
    if len(vista) == 0:
        ax.axis("off"); ax.set_title(classe + " (fora da amostra)", fontsize=11); continue
    desenhar(ax, vista.take(1, seed=SEMENTE).first(), classe)
plt.tight_layout(); plt.show()

**Conferência da orientação:** fotos de celular guardam a rotação num campo EXIF. Se as caixas da grade acima aparecerem deslocadas da planta, mude `APLICAR_EXIF` para `True` nas configurações e rode de novo a partir da seção 3. A célula abaixo conta quantas fotos da amostra têm rotação gravada.

In [ ]:
rotacionadas = sum(1 for s in dataset if Image.open(s.filepath).getexif().get(274, 1) != 1)
print(f"{rotacionadas} de {len(dataset)} imagens têm rotação EXIF")

In [ ]:
# Tamanho relativo das caixas: área da caixa dividida pela área da imagem
areas = []
for s in dataset.select_fields("ground_truth"):
    for d in s.ground_truth.detections:
        areas.append({"classe": d.label, "area_relativa_%": 100 * d.bounding_box[2] * d.bounding_box[3]})
areas = pd.DataFrame(areas)
areas.groupby("classe")["area_relativa_%"].median().sort_values().round(2).to_frame("mediana da área (%)")

As classes no topo desta tabela ocupam uma fração mínima da imagem. É o problema do Carpetweed no slide 10: poucos pixels contra um fundo ruidoso.

## 4. O que é data augmentation (slide 11)

Augmentation cria variações de uma imagem de treino. As **geométricas** mudam a posição (e as caixas precisam acompanhar). As **fotométricas** mudam a aparência (e as caixas ficam iguais).

In [ ]:
exemplo = dataset.filter_labels("ground_truth", F("label") == "Waterhemp").take(1, seed=1).first()
img0 = abrir(exemplo.filepath).convert("RGB"); img0.thumbnail((700, 700))
caixas0 = [d.bounding_box for d in exemplo.ground_truth.detections]

def espelhar_h(c): return [[1 - x - w, y, w, h] for x, y, w, h in c]
def espelhar_v(c): return [[x, 1 - y - h, w, h] for x, y, w, h in c]
def ruido(img):
    a = np.asarray(img).astype(np.float32)
    return Image.fromarray(np.clip(a + np.random.normal(0, 25, a.shape), 0, 255).astype(np.uint8))

variacoes = [
    ("Original", img0, caixas0),
    ("Horizontal flip", img0.transpose(Image.FLIP_LEFT_RIGHT), espelhar_h(caixas0)),
    ("Vertical flip", img0.transpose(Image.FLIP_TOP_BOTTOM), espelhar_v(caixas0)),
    ("Gaussian noise", ruido(img0), caixas0),
    ("Brightness contrast", ImageEnhance.Contrast(ImageEnhance.Brightness(img0).enhance(1.4)).enhance(0.7), caixas0),
    ("Blur", img0.filter(ImageFilter.GaussianBlur(4)), caixas0),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (nome, im, cx) in zip(axes.flat, variacoes):
    w, h = im.size
    ax.imshow(im); ax.axis("off"); ax.set_title(nome, fontsize=12)
    for x, y, bw, bh in cx:
        ax.add_patch(patches.Rectangle((x * w, y * h), bw * w, bh * h, fill=False, edgecolor="cyan", linewidth=2))
plt.tight_layout(); plt.show()

## 5. Split e formatação dos dados (slide 6)

Duas etapas do pipeline:

1. **Split aleatório** em treino, validação e teste (70%, 15%, 15%). A semente define qual divisão sai; trocar a semente é a base da validação de Monte-Carlo.
2. **Formatação:** as anotações originais estão no formato VIA. Exportamos para o formato YOLO, um arquivo `.txt` por imagem com `classe x_centro y_centro largura altura`.

As fotos originais têm até 6.000 pixels de largura. Reduzimos para no máximo 1.280 pixels para o treino caber no tempo da aula. Como as coordenadas YOLO são relativas, as caixas continuam válidas.

In [ ]:
def preparar_split(semente, pasta):
    dataset.untag_samples(["train", "val", "test"])
    four.random_split(dataset, {"train": 0.70, "val": 0.15, "test": 0.15}, seed=semente)
    if os.path.exists(pasta):
        shutil.rmtree(pasta)
    for split in ["train", "val", "test"]:
        dataset.match_tags(split).export(
            export_dir=pasta, dataset_type=fo.types.YOLOv5Dataset,
            label_field="ground_truth", split=split, classes=CLASSES)
    for caminho in glob.glob(f"{pasta}/images/*/*"):
        im = abrir(caminho).convert("RGB")
        im.thumbnail((1280, 1280))
        im.save(caminho, quality=92)   # salva sem EXIF, mantendo a orientação usada nas anotações
    return f"{pasta}/dataset.yaml"

DATA_YAML = preparar_split(SEMENTE, f"{BASE}/dados_seed{SEMENTE}")
print(open(DATA_YAML).read())
for split in ["train", "val", "test"]:
    print(split, len(glob.glob(f"{BASE}/dados_seed{SEMENTE}/images/{split}/*")), "imagens")

In [ ]:
# Como fica um arquivo de anotação no formato YOLO
rotulo = sorted(glob.glob(f"{BASE}/dados_seed{SEMENTE}/labels/train/*.txt"))[0]
print(os.path.basename(rotulo)); print(open(rotulo).read())

## 6. Treinar os modelos (slides 6 e 8)

Usamos **transfer learning**: cada modelo começa com pesos já treinados no COCO (o dataset urbano citado nos slides 7 e 8) e é ajustado às 12 classes de daninhas.

O artigo comparou 25 modelos, do YOLOv3 ao YOLOv7. A biblioteca Ultralytics usada aqui não traz o YOLOv4, então comparamos tamanhos diferentes do YOLOv5, que é suficiente para mostrar o trade-off entre complexidade e velocidade do slide 9. Os modelos com sufixo `u` são as versões atualizadas do YOLOv5 na Ultralytics.

Durante o treino, a Ultralytics já aplica augmentation (mosaic, flips, cor, escala) por padrão.

In [ ]:
from ultralytics import YOLO

resultados = {}
for nome_modelo in MODELOS:
    tag = nome_modelo.replace(".pt", "")
    modelo = YOLO(nome_modelo)
    modelo.train(data=DATA_YAML, epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH, seed=SEMENTE,
                 project=f"{BASE}/runs", name=tag, exist_ok=True, plots=False, verbose=False)
    resultados[tag] = {"pasta": f"{BASE}/runs/{tag}"}
    print("Treino concluído:", tag)

## 7. Acurácia e velocidade (slides 8 e 9)

Avaliamos cada modelo no conjunto de **teste**, que ele nunca viu no treino.

* **mAP@0.5** (acurácia padrão): a caixa prevista precisa cobrir pelo menos 50% da caixa real.
* **mAP@[0.5:0.95]** (acurácia rigorosa): o critério vai de 50% a 95% de sobreposição.
* **Tempo de inferência**: milissegundos por imagem nesta GPU.
* **Parâmetros e GFLOPs**: tamanho e custo de cálculo do modelo.

In [ ]:
linhas = []
for tag, r in resultados.items():
    melhor = YOLO(f"{r['pasta']}/weights/best.pt")
    m = melhor.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
    r["modelo"], r["metricas"] = melhor, m
    info = melhor.info(verbose=False)   # (camadas, parâmetros, gradientes, GFLOPs)
    linhas.append({"modelo": tag,
                   "mAP@0.5 (%)": round(100 * m.box.map50, 2),
                   "mAP@[0.5:0.95] (%)": round(100 * m.box.map, 2),
                   "inferência (ms/img)": round(m.speed["inference"], 2),
                   "parâmetros (milhões)": round(info[1] / 1e6, 2),
                   "GFLOPs": round(info[3], 1)})
comparacao = pd.DataFrame(linhas).set_index("modelo")
comparacao

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for tag, row in comparacao.iterrows():
    axes[0].scatter(row["parâmetros (milhões)"], row["GFLOPs"], s=120)
    axes[0].annotate(tag, (row["parâmetros (milhões)"], row["GFLOPs"]), xytext=(6, 6), textcoords="offset points")
    axes[1].scatter(row["inferência (ms/img)"], row["mAP@0.5 (%)"], s=120)
    axes[1].annotate(tag, (row["inferência (ms/img)"], row["mAP@0.5 (%)"]), xytext=(6, 6), textcoords="offset points")
axes[0].set(title="Complexidade computacional", xlabel="Parâmetros (milhões)", ylabel="GFLOPs")
axes[1].set(title="Eficiência vs. precisão", xlabel="Tempo de inferência (ms)", ylabel="mAP@0.5 (%)")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Como ler:** o canto superior esquerdo do gráfico da direita é o ideal (preciso e rápido). Modelos maiores custam mais GFLOPs e mais milissegundos; a pergunta do slide 9 é se o ganho de mAP compensa esse custo num trator a 20 km/h.

## 8. Acerto por classe (slide 10)

O mAP geral esconde diferenças grandes entre espécies. Aqui vemos o AP@0.5 de cada classe no melhor modelo.

In [ ]:
melhor_tag = comparacao["mAP@0.5 (%)"].idxmax()
m = resultados[melhor_tag]["metricas"]
ap_classe = pd.Series({m.names[i]: 100 * ap for i, ap in zip(m.box.ap_class_index, m.box.ap50)}).sort_values()

cores = ["#B5532A" if c in ("Carpetweed", "Goosegrass") else "#2F5D3A" for c in ap_classe.index]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ap_classe.index, ap_classe.values, color=cores)
for i, v in enumerate(ap_classe.values): ax.text(v + 0.5, i, f"{v:.1f}", va="center")
ax.set(xlabel="AP@0.5 (%)", title=f"Acerto por classe no teste ({melhor_tag})", xlim=(0, 105))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

pd.DataFrame({"AP@0.5 (%)": ap_classe.round(1), "caixas na base completa": pd.Series(completo)}).dropna().sort_values("AP@0.5 (%)")

Compare as duas colunas: as classes com menos anotações e com plantas menores tendem a ficar embaixo. No artigo, Carpetweed ficou em torno de 72% e Goosegrass em torno de 79%. Com amostra pequena, algumas classes raras podem ter pouquíssimos exemplos no teste, então o valor delas aqui oscila bastante.

## 9. Com e sem augmentation (slide 11)

Treinamos o modelo nano de novo, agora **desligando** todas as transformações, e comparamos a curva de mAP por época com o treino original.

In [ ]:
if RODAR_AUGMENTATION:
    sem_aug = dict(hsv_h=0, hsv_s=0, hsv_v=0, degrees=0, translate=0, scale=0, shear=0,
                   perspective=0, flipud=0, fliplr=0, mosaic=0, mixup=0, copy_paste=0)
    YOLO("yolov5nu.pt").train(data=DATA_YAML, epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH, seed=SEMENTE,
                              project=f"{BASE}/runs", name="yolov5nu_sem_aug", exist_ok=True,
                              plots=False, verbose=False, **sem_aug)

    def curva(pasta):
        df = pd.read_csv(f"{pasta}/results.csv"); df.columns = df.columns.str.strip()
        return df["epoch"], 100 * df["metrics/mAP50(B)"]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(*curva(f"{BASE}/runs/yolov5nu"), lw=2.5, color="#1B2A4A", label="Com augmentation")
    ax.plot(*curva(f"{BASE}/runs/yolov5nu_sem_aug"), lw=2.5, ls=":", color="gray", label="Sem augmentation")
    ax.set(xlabel="Épocas", ylabel="mAP@0.5 na validação (%)", title="Efeito do augmentation (YOLOv5n)")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

Com poucos dados, a diferença costuma aparecer bem. O ponto do slide 11 é que, nos modelos modernos, o ganho de **acrescentar** augmentation por fora é pequeno, porque o treino já traz mosaic e outras transformações embutidas. Aqui estamos desligando essas transformações embutidas para mostrar o quanto elas fazem.

## 10. Validação cruzada de Monte-Carlo (slide 6)

Uma única divisão dos dados pode dar sorte. O artigo repetiu o experimento 5 vezes com sementes diferentes e reportou a média. Aqui fazemos o mesmo com 3 sementes e o modelo nano (deixe `RODAR_MONTE_CARLO = True` nas configurações).

In [ ]:
if RODAR_MONTE_CARLO:
    mc = []
    for semente in [1, 2, 3]:
        yaml_mc = preparar_split(semente, f"{BASE}/dados_seed{semente}")
        nome = f"mc_seed{semente}"
        YOLO("yolov5nu.pt").train(data=yaml_mc, epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH, seed=semente,
                                  project=f"{BASE}/runs", name=nome, exist_ok=True, plots=False, verbose=False)
        met = YOLO(f"{BASE}/runs/{nome}/weights/best.pt").val(data=yaml_mc, split="test", plots=False, verbose=False)
        mc.append({"semente": semente, "mAP@0.5 (%)": 100 * met.box.map50, "mAP@[0.5:0.95] (%)": 100 * met.box.map})
    mc = pd.DataFrame(mc).set_index("semente")
    display(mc.round(2))
    print("Média ± desvio do mAP@0.5:", round(mc["mAP@0.5 (%)"].mean(), 2), "±", round(mc["mAP@0.5 (%)"].std(), 2))
else:
    print("Monte-Carlo desligado. Mude RODAR_MONTE_CARLO para True nas configurações e rode de novo esta célula.")

A variação entre sementes mostra por que um único split pode enganar: o desvio é a margem de sorte que a média de várias repetições elimina.

## 11. Predições lado a lado (slide 8)

Anotação real (esquerda) contra a predição do melhor modelo e do menor modelo, em imagens de teste.

In [ ]:
pasta_teste = f"{BASE}/dados_seed{SEMENTE}/images/test"
imagens_teste = sorted(glob.glob(f"{pasta_teste}/*"))
escolhidas = random.Random(SEMENTE).sample(imagens_teste, min(3, len(imagens_teste)))
menor_tag = comparacao["parâmetros (milhões)"].idxmin()

def caixas_reais(caminho):
    rot = caminho.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if not os.path.exists(rot): return []
    return [list(map(float, l.split())) for l in open(rot) if l.strip()]

fig, axes = plt.subplots(len(escolhidas), 3, figsize=(16, 5 * len(escolhidas)))
axes = np.atleast_2d(axes)
for linha, caminho in zip(axes, escolhidas):
    im = Image.open(caminho); w, h = im.size
    linha[0].imshow(im); linha[0].set_title("Anotação real"); linha[0].axis("off")
    for c, xc, yc, bw, bh in caixas_reais(caminho):
        linha[0].add_patch(patches.Rectangle(((xc - bw / 2) * w, (yc - bh / 2) * h), bw * w, bh * h, fill=False, edgecolor="red", lw=2))
        linha[0].text((xc - bw / 2) * w, (yc - bh / 2) * h - 5, CLASSES[int(c)], color="red", fontsize=9)
    for ax, tag in zip(linha[1:], [melhor_tag, menor_tag]):
        pred = resultados[tag]["modelo"].predict(caminho, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
        ax.imshow(pred.plot()[:, :, ::-1]); ax.set_title(f"Predição {tag}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Síntese (slide 12)

O que este notebook mostrou, na mesma ordem da apresentação:

1. **Dados:** o CottonWeedDet12 é grande, mas desequilibrado, e tem plantas que ocupam uma fração mínima da imagem.
2. **Método:** split aleatório, conversão para o formato YOLO, transfer learning a partir do COCO e repetição com sementes diferentes.
3. **Trade-off:** modelos maiores ganham alguns pontos de mAP, mas custam mais GFLOPs e milissegundos. Para robótica embarcada, um modelo pequeno e rápido costuma ser a escolha.
4. **Limites:** as classes raras e de plantas pequenas continuam sendo o ponto fraco, e é aí que entram mais anotações e imagens sintéticas (GANs).

**Referência:** Dang, F., Chen, D., Lu, Y., Li, Z. (2023). YOLOWeeds: A novel benchmark of YOLO object detectors for multi-class weed detection in cotton production systems. *Computers and Electronics in Agriculture*, 205, 107655. https://doi.org/10.1016/j.compag.2023.107655

**Dataset:** https://zenodo.org/records/7535814 · **Código dos autores:** https://github.com/DongChen06/DCW